In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(df.shape)
print(df["TotalCharges"].isna().sum())

(7043, 21)
11


In [2]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print(df.shape)
print(df["TotalCharges"].isna().sum())
print(df["Churn"].value_counts())

(7043, 20)
0
Churn
0    5174
1    1869
Name: count, dtype: int64


In [3]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {count} outliers (normal range {lower:.1f} to {upper:.1f})")

tenure: 0 outliers (normal range -60.0 to 124.0)
MonthlyCharges: 0 outliers (normal range -46.0 to 171.4)
TotalCharges: 0 outliers (normal range -4683.5 to 8868.7)


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(5634, 19) (1409, 19)
0.2653532126375577 0.2654364797728886


In [5]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
binary_cols = ["SeniorCitizen"]
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print(len(categorical_cols), categorical_cols)

15 ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("bin", "passthrough", binary_cols),
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(X_train_prep.shape, X_test_prep.shape)

(5634, 45) (1409, 45)


In [7]:
feature_names = preprocessor.get_feature_names_out()
train_df = pd.DataFrame(X_train_prep, columns=feature_names)

print(len(feature_names))
print(train_df[["num__tenure", "num__MonthlyCharges", "num__TotalCharges"]].mean().round(2))
train_df.iloc[:3, :8]

45
num__tenure           -0.0
num__MonthlyCharges   -0.0
num__TotalCharges      0.0
dtype: float64


,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Female,cat__gender_Male,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No
0,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0
1,-0.711743,0.337478,-0.503635,0.0,1.0,0.0,1.0,0.0
2,-0.793155,-0.809013,-0.749883,0.0,1.0,0.0,1.0,0.0
